# ML for Sensor Health Analyzer

In [25]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

In [26]:
np.random.seed(42)
n = 500

# Features: [value, mean, std, min, max, slope, z_score]
# Simulate realistic temperature sensor readings
value   = np.random.normal(21.0, 1.5, n)      # normal temp around 21°C
mean    = value + np.random.normal(0, 0.2, n)  # mean close to value
std     = np.abs(np.random.normal(1.2, 0.3, n))
min_val = value - np.abs(np.random.normal(2.0, 0.5, n))
max_val = value + np.abs(np.random.normal(2.0, 0.5, n))
slope   = np.random.normal(0.0, 0.02, n)       # mostly stable
z_score = np.random.normal(0.0, 0.5, n)        # low z-score = normal

X_normal = np.column_stack([value, mean, std, min_val, max_val, slope, z_score])
X_normal = X_normal.astype(np.float32)

In [27]:
# ── 2. Build pipeline ─────────────────────────────────────────────────
# Train ONLY on normal data – this is key for IsolationForest
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model",  IsolationForest(
        n_estimators=100,
        contamination=0.05,
        random_state=42,
    )),
])

pipeline.fit(X_normal)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('scaler', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"n_estimators n_estimators: int, default=100The number of base estimators in the ensemble.",100
,"max_samples max_samples: ""auto"", int or float, default=""auto""The number of samples to draw from X to train each base estimator.- If int, then draw `max_samples` samples.- If float, then draw `max_samples * X.shape[0]` samples.- If ""auto"", then `max_samples=min(256, n_samples)`.If max_samples is larger than the number of samples provided,all samples will be used for all trees (no sampling).",'auto'
,"contamination contamination: 'auto' or float, default='auto'The amount of contamination of the data set, i.e. the proportionof outliers in the data set. Used when fitting to define the thresholdon the scores of the samples.- If 'auto', the threshold is determined as in the original paper.- If float, the contamination should be in the range (0, 0.5]... versionchanged:: 0.22 The default value of ``contamination`` changed from 0.1 to ``'auto'``.",0.05
,"max_features max_features: int or float, default=1.0The number of features to draw from X to train each base estimator.- If int, then draw `max_features` features.- If float, then draw `max(1, int(max_features * n_features_in_))` features.Note: using a float number less than 1.0 or integer less than number offeatures will enable feature subsampling and leads to a longer runtime.",1.0


In [28]:
# ── 3. Quick sanity check before export ──────────────────────────────
sample = np.array([[21.0, 21.0, 1.5, 19.0, 23.0, 0.01, 0.1]], dtype=np.float32)
label  = pipeline.predict(sample)   # 1 = normal, -1 = anomaly
score  = pipeline.decision_function(sample)
print(f"Sample label: {label}, score: {score}")

Sample label: [1], score: [0.17074609]


In [29]:
# ── 4. Export to ONNX ─────────────────────────────────────────────────
# n_features must match your feature vector length
initial_type = [("float_input", FloatTensorType([None, 7]))]

onnx_model = convert_sklearn(
    pipeline,
    initial_types=initial_type,
    target_opset={
        "": 17,           # default ONNX opset
        "ai.onnx.ml": 3,  # ML opset version
    },
)

with open("sensor_health.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

print("Exported to sensor_health.onnx")

Exported to sensor_health.onnx


In [30]:
# ── 5. Verify ONNX output names (copy these into Rust) ───────────────
import onnxruntime as rt
sess = rt.InferenceSession("sensor_health.onnx")

print("Inputs:")
for i in sess.get_inputs():
    print(f"  name={i.name} shape={i.shape} type={i.type}")

print("Outputs:")
for o in sess.get_outputs():
    print(f"  name={o.name} shape={o.shape} type={o.type}")

Inputs:
  name=float_input shape=[None, 7] type=tensor(float)
Outputs:
  name=label shape=[None, 1] type=tensor(int64)
  name=scores shape=[None, 1] type=tensor(float)


In [32]:
test_normal  = np.array([[21.0, 21.0, 1.5, 19.0, 23.0, 0.01, 0.1]], dtype=np.float32)
test_anomaly = np.array([[45.0, 21.0, 1.5, 19.0, 45.0, 2.30, 19.9]], dtype=np.float32)

print("normal  score:", pipeline.decision_function(test_normal))
print("anomaly score:", pipeline.decision_function(test_anomaly))
print("normal  label:", pipeline.predict(test_normal))
print("anomaly label:", pipeline.predict(test_anomaly))

scores = pipeline.decision_function(X_normal)
print(f"min:  {scores.min():.4f}")
print(f"max:  {scores.max():.4f}")
print(f"mean: {scores.mean():.4f}")
print(f"p5:   {np.percentile(scores, 5):.4f}")   # 5th percentile
print(f"p95:  {np.percentile(scores, 95):.4f}")  # 95th percentile

normal  score: [0.17074609]
anomaly score: [-0.11881157]
normal  label: [1]
anomaly label: [-1]
min:  -0.1588
max:  0.1812
mean: 0.1085
p5:   0.0000
p95:  0.1675
